# Loan Default Risk Prediction
**Logistic Regression + XGBoost on LendingClub-style data**

This notebook walks through the full credit risk pipeline:
1. Data exploration & EDA
2. Feature engineering
3. Preprocessing pipeline (imputation, encoding, scaling)
4. Logistic Regression with hyperparameter tuning
5. XGBoost with hyperparameter tuning
6. Model evaluation & comparison
7. Stakeholder visualisations — risk scorecard & feature importance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import sys; sys.path.insert(0, '../src')
from loan_default_pipeline import (
    generate_data, engineer_features, build_preprocessor,
    train_logistic_regression, train_xgboost,
    evaluate_model, plot_eda, plot_roc_pr,
    plot_feature_importance, plot_risk_scorecard,
    plot_logistic_coefficients, plot_model_comparison
)
from sklearn.model_selection import train_test_split

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

## 1. Load & Explore Data

In [ ]:
df = generate_data(n=15000)
print(f'Shape: {df.shape}')
print(f'Default rate: {df["loan_default"].mean():.1%}')
df.head()

In [ ]:
df.info()
print('\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

In [ ]:
# Class imbalance
counts = df['loan_default'].value_counts()
fig, ax = plt.subplots(figsize=(5,4))
ax.bar(['Fully Paid','Default'], counts, color=['#2563EB','#DC2626'])
ax.set_title('Class Distribution')
for i, v in enumerate(counts):
    ax.text(i, v+50, f'{v:,}\n({v/len(df):.1%})', ha='center')
plt.tight_layout(); plt.show()

In [ ]:
# EDA overview
plot_eda(df)
from IPython.display import Image
Image('../outputs/01_eda_overview.png')

## 2. Feature Engineering

In [ ]:
df_eng = engineer_features(df)
new_features = [c for c in df_eng.columns if c not in df.columns]
print('New features:', new_features)
df_eng[new_features].describe()

## 3. Train / Test Split

In [ ]:
X = df_eng.drop(columns=['loan_default','sub_grade'])
y = df_eng['loan_default']

num_features = X.select_dtypes(include=['number']).columns.tolist()
cat_features = X.select_dtypes(include=['object']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train default rate: {y_train.mean():.1%}')

## 4. Train Logistic Regression (with tuning)

In [ ]:
lr_model = train_logistic_regression(
    build_preprocessor(num_features, cat_features),
    X_train, y_train
)

## 5. Train XGBoost (with tuning)

In [ ]:
xgb_model = train_xgboost(
    build_preprocessor(num_features, cat_features),
    X_train, y_train
)

## 6. Evaluation & Visualisations

In [ ]:
evaluate_model(lr_model,  X_test, y_test, 'Logistic Regression')
evaluate_model(xgb_model, X_test, y_test, 'XGBoost')

In [ ]:
plot_roc_pr(lr_model, xgb_model, X_test, y_test)
Image('../outputs/02_roc_pr_curves.png')

In [ ]:
plot_feature_importance(xgb_model, num_features, cat_features)
Image('../outputs/04_feature_importance.png')

In [ ]:
plot_risk_scorecard(xgb_model, X_test, y_test)
Image('../outputs/05_risk_scorecard.png')

In [ ]:
plot_logistic_coefficients(lr_model, num_features, cat_features)
Image('../outputs/06_lr_coefficients.png')

In [ ]:
plot_model_comparison(lr_model, xgb_model, X_test, y_test)
Image('../outputs/07_model_comparison.png')